# 03_two_stage default/clf — fit (Stage 1 분류, MODEL_NAME 스위치)

`default/clf/hpo.py --model {MODEL_NAME}` 병렬 HPO study에서 best를 로드해 K-fold refit →
die-level 확률(prob) + unit(평균확률·pred=prob×y_pos_const) 번들 저장. combine이 reg와 곱한다.

- 전처리: `zit_pp.load_for_fit`. refit/저장: `modules.hpo.refit_clf_best` + `save_clf_artifacts`.
- die CSV는 raw `prob`, best_params.json에 `y_pos_const`(=E[Y|Y>0]) 박제 → stacking이 prob×y_pos_const로 health 스케일 변환.
- 산출물: §5.1 의미 폴더 `4_output/03_two_stage/default/clf/{MODEL_NAME}/`.

## 1. 환경 설정

In [1]:
import os, sys
RESUME = True   # 기존 Optuna study(db)에 이어서 학습할지 (필요시 config 셀에서 덮어씀)
# ── Colab이면 코드 번들 1개(code.zip)만 받아 풀기 — 데이터·경로·폰트는 setup.py가 처리 ──
try:
    import google.colab  # Colab에서만 import 성공
    GDRIVE_CODE_ID = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py+requirements+utils+2_preprocessing+3_modeling 지원코드
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip -q install gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
    os.chdir('/content/project')
except ImportError:
    pass
# ── 공통: cwd에서 위로 setup.py(+utils/)를 자동탐색해 실행 (노트북 깊이·드라이브 위치 무관) ──
_d = os.getcwd()
while not (os.path.exists(os.path.join(_d, 'setup.py')) and os.path.isdir(os.path.join(_d, 'utils'))):
    _p = os.path.dirname(_d)
    if _p == _d:
        raise RuntimeError('프로젝트 루트(setup.py + utils/)를 못 찾음 — cwd 확인')
    _d = _p
if _d not in sys.path:
    sys.path.insert(0, _d)
import runpy
runpy.run_path(os.path.join(_d, 'setup.py'))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 공통 유틸: 경로 상수(OUTPUT_DIR, DATA_DIR), 컬럼 상수(TARGET_COL, KEY_COL), SEED
from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing) 경로 + `from modules import ...` 가 3_modeling/modules를 찾게
# 전처리 모듈(2_preprocessing/)과 모델링 모듈(3_modeling/modules/) 모두 sys.path 등록
# 노트북 위치가 달라도 PROJECT_ROOT 기준 절대경로로 접근 → Colab·로컬 동일
PREP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PREP_ROOT not in sys.path:
    sys.path.insert(0, PREP_ROOT)
MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

# preprocess.run: 전체 전처리 파이프라인(결측/이상치/스케일/집계)
# hpo: HPO(run_hpo) + 재학습(refit_best) + 산출물 저장(save_artifacts)
# models: 모델 레지스트리 (AVAILABLE_MODELS 리스트 + 모델 생성 팩토리)
from modules import preprocess, hpo, models   # noqa: E402  (preprocess.run, hpo.run_hpo/refit_best/save_artifacts, models 레지스트리)
# meta_features: position·die_xy 메타피처 생성 (run_wf_xy 파싱 기반)
from meta_features import add_meta_features

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available models: {models.AVAILABLE_MODELS}')

setup 완료


PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트
Available models: ['lgbm', 'xgb', 'catboost', 'et', 'enet', 'zitboost']


## 2. 실험 설정 — MODEL_NAME 스위치 + 소스 study + 후처리

In [ ]:
# 모델 선택 — MODEL_NAME만 바꾸면 4종 모두 처리.
MODEL_NAME = 'lgbm'   # {'lgbm','xgb','catboost','et'}
USER = 'jh'

# 소스 study — default/clf/hpo.py --model {MODEL_NAME} 가 만든 것.
SOURCE_STUDY_NAME = f'ts_clf_{MODEL_NAME}'
SOURCE_DB_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'clf', MODEL_NAME)
SOURCE_DB_PATH = os.path.join(SOURCE_DB_DIR, f'optuna_{USER}_{SOURCE_STUDY_NAME}.db')

N_FOLDS = 5
N_JOBS = -1
CLIP_Y_EXTREME = True

# 산출물 위치 (§5.1). combine이 이 leaf에서 *_die.csv(prob)를 읽는다.
EXP_ID = SOURCE_STUDY_NAME
OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'clf', MODEL_NAME)
os.makedirs(OUT_DIR, exist_ok=True)
print(f'MODEL_NAME={MODEL_NAME} (Stage1 분류) | study={SOURCE_STUDY_NAME}')
print(f'OUT_DIR={OUT_DIR}')

## 3. best trial 로드 + 데이터 (통일 PP)

In [ ]:
import ast
from pathlib import Path
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
from modules import hpo  # refit_clf_best / save_clf_artifacts

_ZIT_DIR = os.path.join(PROJECT_ROOT, '3_modeling', '01_zit')
if _ZIT_DIR not in sys.path:
    sys.path.insert(0, _ZIT_DIR)
import zit_pp


def _parse_user_attr(v):
    if isinstance(v, str):
        try:
            return ast.literal_eval(v)
        except (ValueError, SyntaxError):
            return v
    return v


_storage = f'sqlite:///{Path(SOURCE_DB_PATH).as_posix()}'
_study = optuna.load_study(study_name=SOURCE_STUDY_NAME, storage=_storage)
_best = _study.best_trial
_ua = {k: _parse_user_attr(v) for k, v in _study.user_attrs.items()}
best_params = dict(_best.params)
study_meta_for_save = {
    'exp_id': EXP_ID, 'model_name': MODEL_NAME, 'best_trial_number': _best.number,
    'best_oof_rmse': float(_best.value), 'study_meta': dict(_ua),
}
print(f'[best] study={SOURCE_STUDY_NAME} trial#{_best.number} oof={_best.value:.9f}')

# 데이터 — 통일 PP.
_d = zit_pp.load_for_fit(clip_y_extreme=CLIP_Y_EXTREME)
xs_train, xs_val, xs_test = _d['xs_train'], _d['xs_val'], _d['xs_test']
ys_input = _d['ys_input']
feat_cols_clean = _d['feat_cols']

# y_pos_const = E[Y|Y>0] (train 기준) — clf 확률을 health 스케일로 변환할 상수. best_params.json에 박제.
_yt = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_pos_const = float(_yt[_yt > 0].mean())
print(f'[data] feat={len(feat_cols_clean)}, y_pos_const(E[Y|Y>0])={y_pos_const:.6f}')

## 4. Best refit (K-fold OOF)

In [ ]:
# best HP로 K-fold 재학습 -> die-level 확률 OOF/val/test (val·test는 fold 평균).
final = hpo.refit_clf_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    best_params=best_params,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
)
# die 확률 OOF → unit 평균확률 × y_pos_const → unit RMSE (후처리 이전).
# refit_clf_best는 die-level 'oof_proba_die'만 반환하므로, save_clf_artifacts와 동일하게
# 여기서 die→unit 평균 집계 후 y_pos_const(E[Y|Y>0])를 곱해 health 스케일로 환산한다.
y_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
oof_proba_unit = (pd.Series(final['oof_proba_die'], index=xs_train[KEY_COL].values)
                  .groupby(level=0).mean())
oof_u = (oof_proba_unit * y_pos_const).reindex(y_true.index)
print(f'[Refit 완료] OOF unit RMSE = {float(np.sqrt(np.mean((oof_u.values - y_true.values) ** 2))):.6f}')

## 5. 후처리 + 산출물 저장

In [ ]:
# die.csv(prob) + unit.csv + fold_models.pkl + best_params.json(y_pos_const 포함) 저장.
hpo.save_clf_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    y_pos_const=y_pos_const,
    study_meta=study_meta_for_save,
)
for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:34s}  {size_kb:10,.1f} KB')